# BioMS Zaku — start here

Run every cell (in Colab: **Runtime → Run all**). There is nothing to download and no path to fix:
the example data travel inside the package.

In about a minute you see the three things the tool does:

1. **Decomposition** — the redundancy between two indices, predicted from algebra *before either index is computed*.
2. **Audit** — whether an index measures what it claims, or only tracks a negative control.
3. **The report** — one self-contained file with every number, table and figure of the run.

The example data are **synthetic**: 400 people drawn from the means and covariances of NHANES, per sex × diabetes.
Nobody in this file is a real person, and every number below is reproducible.

In [ ]:
%pip install -q bioms-zaku

## The data come with the package

`load_example` reads the file from inside the installed package. No download, no upload, no path.

In [ ]:
from bioms_zaku.datasets import load_example

df = load_example("zaku_exemplo")
print(df.shape)
df.head()

What the columns are:

| column | meaning |
|---|---|
| `R`, `Xc` | resistance and reactance, bioimpedance at 50 kHz |
| `H_cm`, `W` | stature (cm) and body mass (kg) |
| `idade`, `sexo` | age in years; sex, `0` = F, `1` = M |
| `BMXARMC`, `BMXWAIST`, `BMXCALF` | arm, waist and calf circumferences |
| `LMI_DXA`, `ALMI_DXA`, `FMI_DXA` | lean, appendicular lean and fat mass indices measured by DXA — the reference |
| `label_synthetic` | a label built from fat mass and age **only**: a known answer to check the tool against |
| `diabetes` | doctor-diagnosed diabetes, as in the source survey |

## 1. The algebra: redundancy predicted before anything is computed

An index that is a product of powers is a **vector of exponents**. Over the variables `R, Xc, H, W`:

* impedance index `H² / R` → `(-1, 0, +2, 0)`
* body mass index `W / H²` → `(0, 0, -2, +1)`

Let **Σ** be the covariance of the *logarithms* of the measured variables in a population. Then, for any two such
indices with vectors **a** and **b**, the Pearson correlation of their logs is

$$\rho = \frac{a^{\top}\Sigma b}{\sqrt{a^{\top}\Sigma a \cdot b^{\top}\Sigma b}}$$

an identity that holds for any distribution. Neither index has to be computed. Below, the prediction is compared
with what the data actually show.

In [ ]:
import numpy as np
from scipy.stats import spearmanr
from bioms_zaku.algebra import log_covariance, predicted_pearson_log, pearson_to_spearman

variables = ["R", "Xc", "H_cm", "W"]      # the measured variables, in this order
ii  = np.array([-1, 0,  2, 0])            # H² / R  — impedance index
bmi = np.array([ 0, 0, -2, 1])            # W / H²  — body mass index

for code_, label in ((0, "women"), (1, "men")):
    d = df[df.sexo == code_]
    sigma, n = log_covariance(d, variables)
    predicted = pearson_to_spearman(predicted_pearson_log(ii, bmi, sigma))
    observed = spearmanr(d.H_cm ** 2 / d.R, d.W / d.H_cm ** 2).statistic
    print(f"{label:>6} (n={n}):  predicted {predicted:+.3f}   observed {observed:+.3f}   difference {abs(predicted - observed):.3f}")

The prediction came from Σ alone. Now the same two indices with women and men **mixed together**:

In [ ]:
sigma, n = log_covariance(df, variables)
predicted = pearson_to_spearman(predicted_pearson_log(ii, bmi, sigma))
observed = spearmanr(df.H_cm ** 2 / df.R, df.W / df.H_cm ** 2).statistic
print(f"mixed  (n={n}):  predicted {predicted:+.3f}   observed {observed:+.3f}")

Half of the within-sex value — and the algebra predicts that too. Redundancy is a property of **the population**,
not of the formulas: mixing two populations changes Σ, and with it every correlation. This is why every number below
is computed inside a stratum and never across.

## 2. The audit: does the index measure what it claims?

The question is not whether an index correlates with lean mass — almost everything does. It is whether it predicts
lean mass **beyond** what a fat mass index already predicts. So the run declares, before seeing any result:

* **target**: `LMI_DXA`, lean mass index by DXA
* **negative control**: `FMI_DXA`, fat mass index by DXA

An index is called *specific* only when it beats the control on the target, and *tracks the control* when its apparent
success is explained by the control. Everything is declared in the configuration below — nothing is chosen after the fact.

In [ ]:
from bioms_zaku.datasets import example_path
from bioms_zaku.check import check
from bioms_zaku.run import run

config = {
    "run_name": "notebook",
    "data": {
        "path": str(example_path("zaku_exemplo")),          # your own CSV goes here
        "columns": {
            "variables": {"R": "R", "Xc": "Xc", "H": "H_cm", "W": "W"},
            "units": {"H": "cm", "W": "kg"},
            "covariates": ["W", "H_cm"],
            "groups": {"sexo": "sexo", "idade": "idade"},
            "id": "id",
            "targets": {"LMI_DXA": "LMI_DXA"},
            "controls": {"FMI_DXA": "FMI_DXA"},
            "pairing": {"LMI_DXA": "FMI_DXA"},
        },
    },
    "strata": "sexo",
    "strata_labels": {0: "F", 1: "M"},
    "catalog": {"include": ["Lukaski1985_II", "Baumgartner1988_PhA", "LMI"]},
    "declarations": {
        "targets_independent_of_variables": True,           # DXA is not computed from R, Xc, H, W
        "target_kinds": {"LMI_DXA": "lean_mass", "FMI_DXA": "fat_mass"},
    },
    "preset": "quick",                                      # "full" for the publication-grade run
    "output": {"dir": "./zaku_out", "figures": True},
}

check(config)      # refuses to run on a configuration that cannot answer the question

In [ ]:
res = run(config)

## The verdicts

One row per index and stratum. `redundant`: another index already carries the same information. `specific`: it beats
the negative control on the target. `useful`: it adds something over body mass and stature alone.

In [ ]:
import pandas as pd

pd.read_csv(f"{res['out_dir']}/screening.csv")

## 3. The report, and your own data

`report.html` in the output folder holds every number above with the method text beside it — how it was computed,
how to read it, what rigour was applied — and downloads for every table. It is one file: mail it, archive it, cite it.

To read the same finished run in another language, without recomputing anything:

```python
from bioms_zaku.run import render
render(res["out_dir"], "pt")      # en · es · pt · it
```

For your own data, either point `config["data"]["path"]` at your CSV and edit the column names above, or let the
tool ask you, in a terminal:

```bash
bioms-zaku start meus_dados.csv
```

What the tool guarantees, and under which assumptions, is written in `CONTRATOS.md` in the repository.